# Train Gradient Boost


In [1]:
# Imports
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from sklearn.ensemble import StackingRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, SGDRegressor, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from code_files.train import train, train_in_batches, grid_search, random_search, save_model
from code_files.data_preperation import prepare_for_train
import pandas as pd
import numpy as np
import importlib

In [2]:
# Load Dataset
df_amazon = pd.read_csv("../../dataset/eda_amazon_sales_report.csv")
df_amazon.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117123 entries, 0 to 117122
Data columns (total 24 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   Unnamed: 0                           117123 non-null  int64  
 1   Size                                 117123 non-null  int64  
 2   Qty                                  117123 non-null  int64  
 3   Amount                               117123 non-null  float64
 4   promotion-ids                        117123 non-null  int64  
 5   B2B                                  117123 non-null  int64  
 6   Status_Cancelled                     117123 non-null  bool   
 7   Status_Shipped                       117123 non-null  bool   
 8   Status_Shipped - Delivered to Buyer  117123 non-null  bool   
 9   Fulfilment_Amazon                    117123 non-null  bool   
 10  Fulfilment_Merchant                  117123 non-null  bool   
 11  ship-service-

In [3]:
# Split and Prepare for train
dftrain, dftest = train_test_split(df_amazon, test_size=0.1, random_state=42)
Xtrain_prepared, ytrain_prepared, Xtest_prepared, ytest_prepared = prepare_for_train(dftrain, dftest)

In [4]:

# Grid Search
search = random_search(
    Xtrain_prepared,
    ytrain_prepared,
    StackingRegressor(
        estimators=[
            ('Decision Tree', DecisionTreeRegressor(**{'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2})),
            ('Ridge', Ridge(**{'alpha': 0.1, 'fit_intercept': True, 'solver': 'sag', 'tol': 0.01})),
            ('SGD', SGDRegressor(**{'alpha': 0.0001, 'eta0': 0.001, 'learning_rate': 'invscaling', 'penalty': 'elasticnet'}))
        ]
    ),
    params={
        "final_estimator": [
            GradientBoostingRegressor(),
            RandomForestRegressor(),
            Ridge(),
            Lasso(),
            SVR(),
        ],  # Different meta-models for stacking
        "cv": [3, 5, 10],  # Number of folds in cross-validation
        # Whether to include original features for the meta-model
        "passthrough": [False, True],


    },
    n_iters=30,
    cv=2,
)

save_model(search.best_estimator_)

Fitting 2 folds for each of 30 candidates, totalling 60 fits
{'cv': 10, 'estimators': [('Decision Tree', DecisionTreeRegressor(max_depth=10, min_samples_leaf=2)), ('Ridge', Ridge(alpha=0.1, solver='sag', tol=0.01)), ('SGD', SGDRegressor(eta0=0.001, penalty='elasticnet'))], 'final_estimator__C': 1.0, 'final_estimator__cache_size': 200, 'final_estimator__coef0': 0.0, 'final_estimator__degree': 3, 'final_estimator__epsilon': 0.1, 'final_estimator__gamma': 'scale', 'final_estimator__kernel': 'rbf', 'final_estimator__max_iter': -1, 'final_estimator__shrinking': True, 'final_estimator__tol': 0.001, 'final_estimator__verbose': False, 'final_estimator': SVR(), 'n_jobs': None, 'passthrough': False, 'verbose': 0, 'Decision Tree': DecisionTreeRegressor(max_depth=10, min_samples_leaf=2), 'Ridge': Ridge(alpha=0.1, solver='sag', tol=0.01), 'SGD': SGDRegressor(eta0=0.001, penalty='elasticnet'), 'Decision Tree__ccp_alpha': 0.0, 'Decision Tree__criterion': 'squared_error', 'Decision Tree__max_depth': 1

In [5]:
# Show results
df_grid_results = pd.DataFrame(search.cv_results_)
columns_to_show = ["params", "rank_test_score", "mean_train_score", "mean_test_score"]
df_shown_results = df_grid_results[columns_to_show]

print(search.best_params_)
df_shown_results.sort_values("rank_test_score", ascending = True)

{'passthrough': False, 'final_estimator': SVR(), 'cv': 10}


,params,rank_test_score,mean_train_score,mean_test_score
28,"{'passthrough': False, 'final_estimator': SVR(...",1,-206.709837,-208.137531
18,"{'passthrough': False, 'final_estimator': SVR(...",2,-206.729897,-208.157503
8,"{'passthrough': False, 'final_estimator': SVR(...",3,-206.916204,-208.163834
29,"{'passthrough': True, 'final_estimator': SVR()...",4,-206.985075,-209.084945
19,"{'passthrough': True, 'final_estimator': SVR()...",5,-207.047348,-209.109488
9,"{'passthrough': True, 'final_estimator': SVR()...",6,-207.311501,-209.227890
1,"{'passthrough': True, 'final_estimator': Gradi...",7,-209.318460,-210.644424
11,"{'passthrough': True, 'final_estimator': Gradi...",8,-209.100930,-210.680608
21,"{'passthrough': True, 'final_estimator': Gradi...",9,-209.369747,-210.860291
0,"{'passthrough': False, 'final_estimator': Grad...",10,-209.613436,-211.440639


In [6]:
# Train
model, scores = train(search.best_estimator_, Xtrain_prepared, ytrain_prepared, Xtest_prepared, ytest_prepared)
print(f"mae: {scores[0]}, rmse: {scores[1]}, r2: {scores[2]}")

mae: 208.52851108442425, rmse: 282.2513055647448, r2: 282.2513055647448


Best Param:
{'bootstrap': True, 'max_depth': 10, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 200}